In [1]:
import time
import re
from pymongo import MongoClient
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

# ==========================================================
# 1. CONFIGURACIÓN DE METADATOS Y CONEXIÓN A MONGODB
# ==========================================================
NOMBRE_GRUPO = "G1_Ecommerce" # Reemplaza con el nombre de tu grupo [cite: 320]
CATEGORIA_ACTUAL = "Travel"   # Reemplaza según la categoría de libros [cite: 320]

try:
    # Se ajusta la colección a 'datos_libros' como indica el Cap 6.3 [cite: 320]
    client = MongoClient('mongodb://database:27017/') 
    db = client['proyecto_bigdata']
    coleccion = db['datos_libros']
    print(f"CONEXIÓN EXITOSA: Listo para insertar datos de {CATEGORIA_ACTUAL}")
except Exception as e:
    print("ERROR DE CONEXIÓN:", e)

# ==========================================================
# 2. CONFIGURACIÓN DEL DRIVER (Navegador fantasma)
# ==========================================================
def iniciar_navegador():
    opciones = Options()
    opciones.binary_location = "/usr/bin/google-chrome"
    opciones.add_argument("--headless")
    opciones.add_argument("--no-sandbox")
    opciones.add_argument("--disable-dev-shm-usage")
    opciones.add_argument("--window-size=1920,1080")
    opciones.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    )
    return webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=opciones)

driver = iniciar_navegador()

# ==========================================================
# 3. EXTRACCIÓN DINÁMICA Y ALMACENAMIENTO (El "Gran Salto")
# ==========================================================
url_inicial = "https://books.toscrape.com/"
paginas_objetivo = 5

try:
    print(f"Conectando a {url_inicial}...")
    driver.get(url_inicial)

    for pagina in range(1, paginas_objetivo + 1):
        print(f"--- Procesando Página {pagina} / {paginas_objetivo} ---")

        # AJUSTE 1: Espera Explícita en lugar de time.sleep()
        WebDriverWait(driver, 10).until(
            EC.presence_of_all_elements_located((By.CSS_SELECTOR, "article.product_pod"))
        )

        libros = driver.find_elements(By.CSS_SELECTOR, "article.product_pod")

        for libro in libros:
            try:
                # Extracción de datos básicos
                titulo = libro.find_element(By.CSS_SELECTOR, "h3 a").get_attribute("title").strip()
                precio_texto = libro.find_element(By.CSS_SELECTOR, "p.price_color").text
                
                # Limpieza robusta del valor a número flotante 
                precio_limpio = float(re.sub(r"[^\d.]", "", precio_texto))

                # AJUSTE 2: Diccionario enriquecido para MongoDB 
                documento = {
                    "item": titulo,
                    "valor": precio_limpio,
                    "categoria": CATEGORIA_ACTUAL,
                    "grupo": NOMBRE_GRUPO,
                    "fecha_captura": time.strftime("%Y-%m-%d %H:%M:%S")
                }

                # AJUSTE 3: Almacenamiento Directo fila por fila 
                coleccion.insert_one(documento)

            except Exception as e:
                # Si un elemento falla, lo saltamos silenciosamente para no detener el proceso
                continue 

        # Paginación con Espera Explícita
        if pagina < paginas_objetivo:
            try:
                boton_siguiente = WebDriverWait(driver, 5).until(
                    EC.element_to_be_clickable((By.XPATH, "//li[@class='next']/a"))
                )
                boton_siguiente.click()
                time.sleep(2) # Breve pausa para dar tiempo al renderizado de JavaScript
            except Exception:
                print("  ℹ Botón 'Next' no encontrado. Última página alcanzada.")
                break

    print("\nPROCESO FINALIZADO: Datos enviados a MongoDB.")

except Exception as e:
    print(f"Error detectado durante la ejecución: {e}")

finally:
    # AJUSTE 4: Cierre seguro obligatorio
    driver.quit()
    print("Navegador cerrado de forma segura.")

CONEXIÓN EXITOSA: Listo para insertar datos de Travel
Conectando a https://books.toscrape.com/...
--- Procesando Página 1 / 5 ---
--- Procesando Página 2 / 5 ---
--- Procesando Página 3 / 5 ---
--- Procesando Página 4 / 5 ---
--- Procesando Página 5 / 5 ---

PROCESO FINALIZADO: Datos enviados a MongoDB.
Navegador cerrado de forma segura.
